In [4]:
import pandas as pd

df = pd.read_csv("../data/raw/twcs.csv")

In [5]:
df["in_response_to_tweet_id"] = pd.to_numeric(
    df["in_response_to_tweet_id"],
    errors="coerce"
).astype("Int64")

In [6]:
parent = dict(
    zip(
        df["tweet_id"],
        df["in_response_to_tweet_id"]
    )
)


def find_root(tweet_id):
    visited = set()

    while pd.notna(parent.get(tweet_id)):

        if tweet_id in visited:
            break

        visited.add(tweet_id)

        tweet_id = int(parent[tweet_id])

    return tweet_id


df["conversation_id"] = df["tweet_id"].map(find_root)

In [7]:
amazon_conversation_ids = df[
    df["author_id"] == "AmazonHelp"
]["conversation_id"].dropna().unique()


amazon_df = df[
    df["conversation_id"].isin(amazon_conversation_ids)
].copy()

In [8]:
print("Shape:")
print(amazon_df.shape)

print("\nNumber of unique conversations:")
print(amazon_df["conversation_id"].nunique())

print("\nInbound value counts:")
print(amazon_df["inbound"].value_counts())

print("\nTop 10 authors:")
print(amazon_df["author_id"].value_counts().head(10))

Shape:
(374042, 8)

Number of unique conversations:
82534

Inbound value counts:
inbound
True     203598
False    170444
Name: count, dtype: int64

Top 10 authors:
author_id
AmazonHelp    169840
169172           447
UPSHelp          322
115850           243
Tesco            172
326613           163
158494           119
469296           116
177530            99
270757            86
Name: count, dtype: int64


In [9]:
other_outbound_authors = (
    amazon_df[
        (amazon_df["inbound"] == False) &
        (amazon_df["author_id"] != "AmazonHelp")
    ]["author_id"]
    .value_counts()
)

print("Number of other outbound authors:",
      other_outbound_authors.shape[0])

print(other_outbound_authors.head(30))

Number of other outbound authors: 25
author_id
UPSHelp           322
Tesco             172
XboxSupport        14
hulu_support       11
SpotifyCares       10
JetBlue             9
BofA_Help           8
ChaseSupport        8
AskeBay             7
AppleSupport        6
AskPayPal           6
ArgosHelpers        5
AskSeagate          5
Morrisons           4
DellCares           2
idea_cares          2
AskPlayStation      2
MicrosoftHelps      2
AskAmex             2
AsurionCares        2
Uber_Support        1
Ask_Spectrum        1
AskTarget           1
AldiUK              1
TMobileHelp         1
Name: count, dtype: int64


In [10]:
# Find conversations where UPSHelp appears
ups_conversation_ids = amazon_df[
    amazon_df["author_id"] == "UPSHelp"
]["conversation_id"].unique()

print("Number of conversations involving UPSHelp:",
      len(ups_conversation_ids))

Number of conversations involving UPSHelp: 246


In [11]:
sample_ups_conversations = pd.Series(
    ups_conversation_ids
).sample(3, random_state=42)

for conversation_id in sample_ups_conversations:

    print("=" * 100)
    print(f"Conversation: {conversation_id}")
    print("=" * 100)

    conversation = amazon_df[
        amazon_df["conversation_id"] == conversation_id
    ].copy()

    conversation = conversation.sort_values("created_at")

    print(
        conversation[
            [
                "tweet_id",
                "created_at",
                "author_id",
                "inbound",
                "text"
            ]
        ].to_string(index=False)
    )

    print()

Conversation: 483724
 tweet_id                     created_at  author_id  inbound                                                                                                                                                                                                                                                              text
   483724 Fri Dec 01 14:49:13 +0000 2017     230075     True                                                                 @115817 @115821\n\nI have an order with two day delivery paid option placed on 27th Nov. Still items (tracking #1Z973Y6A0300840635) not delivered. Now I am leaving from town. Will miss my items
   483722 Fri Dec 01 14:55:53 +0000 2017 AmazonHelp    False                                                                                                                     @230075 I'm sorry your items won't be arriving before you leave. We'd like to review your available options here: https://t.co/hApLpMlfHN ^RA
   483726 Fri Dec 01 1

In [12]:
amazon_df = amazon_df.copy()

In [13]:
amazon_df["created_at"] = pd.to_datetime(
    amazon_df["created_at"],
    utc=True
)
print(amazon_df["created_at"].dtype)
print(amazon_df["created_at"].head())

C:\Users\Nihelesh M U\AppData\Local\Temp\ipykernel_6696\1088289157.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  amazon_df["created_at"] = pd.to_datetime(


datetime64[ns, UTC]
181   2017-11-22 09:23:01+00:00
182   2017-11-22 09:24:30+00:00
183   2017-11-22 09:30:36+00:00
184   2017-11-22 09:40:27+00:00
185   2017-11-22 09:44:04+00:00
Name: created_at, dtype: datetime64[ns, UTC]


In [14]:
amazon_sorted = amazon_df.sort_values(
    ["conversation_id", "created_at"]
).copy()

conversation_id = 483724

conversation = amazon_sorted[
    amazon_sorted["conversation_id"] == conversation_id
]

print(
    conversation[
        [
            "conversation_id",
            "tweet_id",
            "created_at",
            "author_id",
            "inbound",
            "text"
        ]
    ].to_string(index=False)
)

 conversation_id  tweet_id                created_at  author_id  inbound                                                                                                                                                                                                                                                              text
          483724    483724 2017-12-01 14:49:13+00:00     230075     True                                                                 @115817 @115821\n\nI have an order with two day delivery paid option placed on 27th Nov. Still items (tracking #1Z973Y6A0300840635) not delivered. Now I am leaving from town. Will miss my items
          483724    483722 2017-12-01 14:55:53+00:00 AmazonHelp    False                                                                                                                     @230075 I'm sorry your items won't be arriving before you leave. We'd like to review your available options here: https://t.co/hApLpMlfHN ^RA
       

In [15]:
amazon_sorted["turn_number"] = (
    amazon_sorted
    .groupby("conversation_id")
    .cumcount() + 1
)

conversation = amazon_sorted[
    amazon_sorted["conversation_id"] == 483724
]

print(
    conversation[
        [
            "conversation_id",
            "turn_number",
            "tweet_id",
            "created_at",
            "author_id",
            "inbound",
            "text"
        ]
    ].to_string(index=False)
)

 conversation_id  turn_number  tweet_id                created_at  author_id  inbound                                                                                                                                                                                                                                                              text
          483724            1    483724 2017-12-01 14:49:13+00:00     230075     True                                                                 @115817 @115821\n\nI have an order with two day delivery paid option placed on 27th Nov. Still items (tracking #1Z973Y6A0300840635) not delivered. Now I am leaving from town. Will miss my items
          483724            2    483722 2017-12-01 14:55:53+00:00 AmazonHelp    False                                                                                                                     @230075 I'm sorry your items won't be arriving before you leave. We'd like to review your available options he

In [16]:
first_turns = (
    amazon_sorted
    .groupby("conversation_id")["turn_number"]
    .min()
)

print(first_turns.value_counts())

turn_number
1    82534
Name: count, dtype: int64


In [17]:
url_count = amazon_sorted["text"].str.contains(
    r"http|https|t\.co",
    case=False,
    regex=True,
    na=False
).sum()

mention_count = amazon_sorted["text"].str.contains(
    r"@\w+",
    regex=True,
    na=False
).sum()

newline_count = amazon_sorted["text"].str.contains(
    r"\n",
    regex=True,
    na=False
).sum()

print("Tweets containing URLs:", url_count)
print("Tweets containing mentions:", mention_count)
print("Tweets containing newlines:", newline_count)

print("Missing text:", amazon_sorted["text"].isna().sum())

print(
    "Empty text:",
    (amazon_sorted["text"].str.strip() == "").sum()
)

Tweets containing URLs: 102691
Tweets containing mentions: 362685
Tweets containing newlines: 21632
Missing text: 0
Empty text: 0


In [18]:
mention_examples = amazon_sorted[
    amazon_sorted["text"].str.contains(
        r"@\w+",
        regex=True,
        na=False
    )
]["text"].head(20)

for text in mention_examples:
    print(text)
    print("-" * 100)

@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET
----------------------------------------------------------------------------------------------------
@AmazonHelp ありがとうございます。
今、電話で主人が対応していただいてます。
----------------------------------------------------------------------------------------------------
@AmazonHelp 電話で対応してもらいましたが改良されませんでした。
保証期間も過ぎてるので買い直しになるんでしょうね。
----------------------------------------------------------------------------------------------------
@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET
----------------------------------------------------------------------------------------------------
@AmazonHelp こちらこそありがとうございました。
----------------------------------------------------------------------------------------------------
@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET
----------------------------------------------------------------------------------------------------
@11

In [19]:
company_mentions = (
    amazon_sorted["text"]
    .str.findall(r"@\w+")
    .explode()
    .value_counts()
)

print(company_mentions.head(30))

text
@AmazonHelp    136680
@115821         22640
@115850         21663
@115830          9232
@115851          3405
@120533          2531
@116928          2097
@116316          1683
@116875          1593
@117086          1509
@119625          1332
@117795           797
@115833           791
@115817           716
@116618           714
@118702           708
@116090           683
@118706           647
@117634           646
@118919           527
@115825           469
@amazonhelp       458
@120540           420
@119356           360
@117093           349
@UPSHelp          340
@116313           339
@116935           303
@132994           286
@137605           283
Name: count, dtype: int64


In [20]:
import re
import html

In [22]:
def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)

    # Remove Twitter agent signatures
    text = re.sub(r"\^[A-Za-z]{1,4}\b", "", text)

    # Remove emojis
    emoji_pattern = re.compile(
        "["
        "\U0001F1E0-\U0001F1FF"
        "\U0001F300-\U0001F5FF"
        "\U0001F600-\U0001F64F"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FAFF"
        "\U00002700-\U000027BF"
        "\U00002600-\U000026FF"
        "]+",
        flags=re.UNICODE
    )

    text = emoji_pattern.sub("", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [23]:
test_texts = [
    "@115770 I have an order problem https://t.co/example",
    "@AmazonHelp My order hasn't arrived 😭",
    "@UPSHelp Please check my package",
    "Hello\n\nI need help with my refund",
    "   My order     is delayed   "
]

for text in test_texts:
    print("BEFORE:", text)
    print("AFTER :", clean_text(text))
    print("-" * 80)

BEFORE: @115770 I have an order problem https://t.co/example
AFTER : @115770 I have an order problem
--------------------------------------------------------------------------------
BEFORE: @AmazonHelp My order hasn't arrived 😭
AFTER : @AmazonHelp My order hasn't arrived
--------------------------------------------------------------------------------
BEFORE: @UPSHelp Please check my package
AFTER : @UPSHelp Please check my package
--------------------------------------------------------------------------------
BEFORE: Hello

I need help with my refund
AFTER : Hello I need help with my refund
--------------------------------------------------------------------------------
BEFORE:    My order     is delayed   
AFTER : My order is delayed
--------------------------------------------------------------------------------


In [24]:
amazon_sorted["clean_text"] = amazon_sorted["text"].apply(clean_text)

amazon_sorted[["text", "clean_text"]].head(20)

,text,clean_text
187,amazonのfireTVstickが見れない😢,amazonのfireTVstickが見れない
181,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...
182,@AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。,@AmazonHelp ありがとうございます。 今、電話で主人が対応していただいてます。
183,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。 保証期間も過ぎて...
184,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...
185,@AmazonHelp こちらこそありがとうございました。,@AmazonHelp こちらこそありがとうございました。
186,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...
235,amazonプライムビデオ、再生エラーが多いです,amazonプライムビデオ、再生エラーが多いです
234,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止>端末の再起動...
325,Way to drop the ball on customer service @1158...,Way to drop the ball on customer service @1158...


In [25]:
! pip install langdetect


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"

sample = amazon_sorted["clean_text"].dropna().sample(
    n=min(10000, len(amazon_df)),
    random_state=42
)

sample_languages = sample.apply(detect_language)

sample_languages.value_counts()

clean_text
en         7523
ja          541
es          513
fr          469
de          276
pt          182
hu          148
it          117
nl           94
tr           21
tl           16
unknown      10
ca            9
af            8
pl            7
no            7
hr            6
hi            6
so            5
id            5
cy            5
ro            4
da            4
sw            4
sl            4
et            4
sv            3
cs            2
sk            2
vi            2
el            1
kn            1
lv            1
Name: count, dtype: int64

In [27]:
remaining_urls = amazon_sorted["clean_text"].str.contains(
    r"http|https|t\.co",
    case=False,
    regex=True,
    na=False
).sum()

print("URLs remaining:", remaining_urls)

remaining_user_mentions = amazon_sorted["clean_text"].str.contains(
    r"@\d+",
    regex=True,
    na=False
).sum()

print("Numeric mentions remaining:", remaining_user_mentions)

remaining_newlines = amazon_sorted["clean_text"].str.contains(
    r"\n",
    regex=True,
    na=False
).sum()

print("Newlines remaining:", remaining_newlines)

empty_clean_text = (
    amazon_sorted["clean_text"]
    .str.strip()
    .eq("")
    .sum()
)

print("Empty cleaned tweets:", empty_clean_text)

sample = amazon_sorted[
    amazon_sorted["text"] != amazon_sorted["clean_text"]
].sample(
    10,
    random_state=42
)

for _, row in sample.iterrows():
    print("ORIGINAL:")
    print(row["text"])
    
    print("\nCLEANED:")
    print(row["clean_text"])
    
    print("\n" + "=" * 100 + "\n")

URLs remaining: 149
Numeric mentions remaining: 247270
Newlines remaining: 0
Empty cleaned tweets: 16
ORIGINAL:
@227093 I am sorry to hear this, Sabrina. When was the book due to be delivered? Have you contacted the seller to highlight the issue with them?^SM

CLEANED:
@227093 I am sorry to hear this, Sabrina. When was the book due to be delivered? Have you contacted the seller to highlight the issue with them?


ORIGINAL:
@AmazonHelp Ayudaaaa
Hice el pedido desde el viernes pero me dicen que les tengo que pasar el número con el que se registró mi tarjeta pero no me marcan ni nada y me urge el paquete 😫

CLEANED:
@AmazonHelp Ayudaaaa Hice el pedido desde el viernes pero me dicen que les tengo que pasar el número con el que se registró mi tarjeta pero no me marcan ni nada y me urge el paquete


ORIGINAL:
@420331 Avez-vous signalé cet incident à notre service client s'il vous plaît? ^BR

CLEANED:
@420331 Avez-vous signalé cet incident à notre service client s'il vous plaît?


ORIGINAL:
@

In [28]:
remaining_urls = amazon_sorted["clean_text"].str.contains(
    r"https?://\S+",
    regex=True,
    na=False
).sum()

print("Actual URLs remaining:", remaining_urls)

Actual URLs remaining: 0


In [29]:
empty_examples = amazon_sorted[
    amazon_sorted["clean_text"].str.strip() == ""
]

print(
    empty_examples[
        [
            "conversation_id",
            "tweet_id",
            "author_id",
            "inbound",
            "text",
            "clean_text"
        ]
    ].to_string(index=False)
)

 conversation_id  tweet_id author_id  inbound                                            text clean_text
           96704     96703    137102     True                         https://t.co/MaiDMbahcK           
          144191    144191    115821     True                         https://t.co/TJv7fBjqGJ           
          279175    279175    182576     True                         https://t.co/MRgqJVCl3x           
          722492    722491    292983     True                         https://t.co/8u9ZmNinDF           
          969291    969291    218503     True                         https://t.co/NyAuiFpvBr           
         1108277   1108276    381559     True                         https://t.co/fFFYfgBHyn           
         1193911   1193911    400536     True                         https://t.co/6AB5B9jL1D           
         1561133   1561133    482291     True                         https://t.co/x1dE72Eo4o           
         1771864   1771864    303368     True https://t

In [30]:
amazon_clean = amazon_sorted[
    amazon_sorted["clean_text"].str.strip() != ""
].copy()

In [31]:
print("Conversations before:", amazon_sorted["conversation_id"].nunique())
print("Conversations after:", amazon_clean["conversation_id"].nunique())

Conversations before: 82534
Conversations after: 82534


In [34]:
from langdetect import detect, DetectorFactory

# Make language detection reproducible
DetectorFactory.seed = 42


print("\nDetecting languages...")

amazon_sorted["language"] = amazon_sorted["clean_text"].apply(
    detect_language
)


print("\nLanguage distribution:")
print(
    amazon_sorted["language"]
    .value_counts()
)

amazon_sorted.to_csv(
    "../data/processed/amazonhelp_cleaned.csv",
    index=False
)

print(
    "\n✅ Complete cleaned dataset saved:"
    " ../data/processed/amazonhelp_cleaned.csv"
)


# ============================================================
# 6. Create English-only dataset
# ============================================================

amazon_english_df = amazon_sorted[
    amazon_sorted["language"] == "en"
].copy()


# ============================================================
# 7. Save English dataset
# ============================================================

amazon_english_df.to_csv(
    "../data/processed/amazonhelp_english.csv",
    index=False
)

print(
    "✅ English dataset saved:"
    " ../data/processed/amazonhelp_english.csv"
)


# ============================================================
# 8. Verification
# ============================================================

print("\n" + "=" * 70)
print("VERIFICATION")
print("=" * 70)

print(
    "\nAll tweets:",
    len(amazon_sorted)
)

print(
    "English tweets:",
    len(amazon_english_df)
)

print(
    "\nAll conversations:",
    amazon_sorted["conversation_id"].nunique()
)

print(
    "English conversations:",
    amazon_english_df["conversation_id"].nunique()
)


# ============================================================
# 9. Check language column
# ============================================================

print("\nLanguage column:")
print(
    amazon_sorted[
        ["tweet_id", "clean_text", "language"]
    ].head(20)
)


# ============================================================
# 10. Check that English dataset contains only English
# ============================================================

print("\nLanguages in English dataset:")

print(
    amazon_english_df["language"]
    .value_counts()
)


Detecting languages...

Language distribution:
language
en         281021
ja          19703
es          19402
fr          18011
de          10327
pt           7185
hu           4970
it           4412
nl           4105
tr            739
unknown       479
tl            469
ca            350
pl            327
ro            316
da            278
af            230
sl            198
et            178
so            177
id            166
hi            146
cy            128
sw            122
no            119
hr             96
sv             60
sk             60
cs             47
vi             45
fi             43
lv             33
lt             31
sq             25
el             22
ko              6
mr              4
ar              3
ta              3
kn              2
ml              1
ne              1
zh-cn           1
fa              1
Name: count, dtype: int64

✅ Complete cleaned dataset saved: ../data/processed/amazonhelp_cleaned.csv
✅ English dataset saved: ../data/processed/amazon